# **Problem Statement**
---



**Writing Viterbi Algorithm for the Primer**

Write the Viterbi algorithm to implment Nature Primer.

Here are some suggestions:

a. You can begin by first defining all the parameters, such as states, transition matrix, and emmision matrix etc.

b. You can write a function to exactly calculate the values mentioned in the primer, for example, you can define a function get_log_prob_of_a_given_path ("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA"). This should output -41.22.

By doing the above two, you earn 1 mark.

Now, you have to implement this in real to get max likely path that would emmit the observed sequence. You MUST note that maximum likely path will just be Es, but that is okay. Implementation is the key.

# **Approach**


---


## **1. Define the HMM Structure**

We start by defining the necessary components of the HMM:

- **States**:  
  The three biological regions:  
  - `E`: Exon  
  - `5`: Donor splice site  
  - `I`: Intron  

- **Initial Probabilities**:  
  The model assumes the sequence always begins in the exon state (`E`), so:  
  - P(E) = 1.0  
  - P(5) = 0.0  
  - P(I) = 0.0

- **Transition Probabilities**:  
  Define the likelihood of moving from one state to another. For example:  
  - `E → E` with high probability  
  - `E → 5` with lower probability  
  - `5 → I` always occurs  
  - `I → I` for intron continuation  
  - `I → end` for termination  

- **Emission Probabilities**:  
  Define the likelihood of each nucleotide (A, C, G, T) being emitted by a state.  
  For instance, exons emit all nucleotides uniformly, while splice sites and introns have distinct preferences (e.g., `5` favors G heavily).

---

## **2. Log-Probability of a Known Path**

Before applying the Viterbi algorithm, we verify the correctness of the model by calculating the **log-probability** of a known path emitting a given sequence.

### Key Steps:
- For each position in the sequence:
  - Multiply the transition probability (from previous state to current)
  - Multiply the emission probability (of the observed nucleotide from current state)
  - Take the logarithm of each (to avoid underflow and allow addition instead of multiplication)
- Sum all the log-probabilities across the path
- If the last state is an intron (`I`), include the log-probability of transitioning to the `end` state

This step validates that our model is behaving as expected and matches the Nature Primer output (e.g., log-probability ≈ -41.22 for a specific example).

---

## **3. Viterbi Algorithm – Dynamic Programming Approach**

The Viterbi algorithm is used to compute the **most likely sequence of hidden states** that could have emitted the given observed DNA sequence.

### Key Concepts:

- **Initialization**:  
  Set up the starting probabilities for each state using the first observation.

- **Recursion (Dynamic Programming)**:  
  For each nucleotide in the sequence:
  - For each current state, evaluate all possible previous states
  - Compute the total log-probability of reaching the current state from each previous one
  - Keep the maximum log-probability and remember which path led to it

- **Path Tracking**:  
  At every step, store the most probable path that leads to each state.

- **Termination**:  
  After the last observation, identify the final state with the highest log-probability
  - Backtrack using the stored paths to reconstruct the full sequence of hidden states

---

## **4. Output and Interpretation**

Once the Viterbi algorithm runs, it produces:

- The **most likely path** of hidden states (e.g., `"EEEEEEEEEEEEEEEEEE5IIIIIII"`)
- The **log-probability** of that path emitting the given observed DNA sequence

Due to the emission probabilities being uniform in exons and heavily skewed elsewhere, the model might favor paths filled with exons (`E`). However, this is expected and acceptable per the Primer.

---

## Summary of Approach

| Step | Description |
|------|-------------|
| 1. | Define states, transition, emission, and initial probabilities |
| 2. | Validate the model by computing the log-probability of a known path |
| 3. | Implement Viterbi using dynamic programming to find the most likely hidden path |
| 4. | Output the most likely state sequence and its log-probability |

---

## Notes

- All probabilities are computed in log-space to prevent numerical underflow.
- Transition to the `end` state is considered only from the intron state.
- This approach allows biological interpretation of gene structure from raw DNA sequences.

In [ ]:
import numpy as np
import math

# Helper function to calculate the log of a probability, avoiding log(0)
def log(x):
    return -math.inf if x == 0 else math.log(x)

# Function to calculate the log probability of a given path and observed sequence
def get_log_prob_of_a_given_path(path: str, seq: str) -> float:
    if len(path) != len(seq):
        raise ValueError("Path and sequence must be of the same length")

    prob = 0.0
    for i in range(len(seq)):
        p1 = path[i]  # State at position i in the path
        s1 = seq[i]   # Observation at position i in the sequence
        if i == 0:
            # Start probability for the first state in the path
            prob += log(start_prob[p1])
        else:
            # Transition probability from the previous state to the current state in the path
            prob += log(trans_prob[path[i-1]][p1])
        # Emission probability for the current state and observation
        prob += log(emit_prob[p1][s1])

    # Transition to End (only possible from 'I' as in Nature Primer)
    last_state = path[-1]
    if last_state == 'I':
        prob += log(trans_prob['I']['end'])  # Transition from 'I' to End

    return prob

# Define all the Parameters as in Nature Primer
states = ['E', '5', 'I']
start_prob = {'E': 1.0, '5': 0.0, 'I': 0.0}

trans_prob = {
    'E': {'E': 0.9, '5': 0.1},  # Transition probabilities from 'E'
    '5': {'I': 1.0},             # Transition probability from '5' to 'I'
    'I': {'I': 0.9, 'end': 0.1},  # Transition probabilities from 'I' to 'I' and 'I' to 'End'
}

emit_prob = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},  # Emission probabilities for 'E'
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},    # Emission probabilities for '5'
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}        # Emission probabilities for 'I'
}

# Test with a given path and sequence
path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

# Calculate the log probability of the given path
log_prob = get_log_prob_of_a_given_path(path, sequence)
print("Log probability of given path:", round(log_prob, 2))

Log probability of given path: -41.22


In [ ]:
import numpy as np
import math

# Helper function to calculate the log of a probability, avoiding log(0)
def log(x):
    return -math.inf if x == 0 else math.log(x)

# Function to calculate the log probability of a given path and observed sequence
def get_log_prob_of_a_given_path(path: str, seq: str) -> float:
    if len(path) != len(seq):
        raise ValueError("Path and sequence must be of the same length")

    prob = 0.0
    for i in range(len(seq)):
        p1 = path[i]  # State at position i in the path
        s1 = seq[i]   # Observation at position i in the sequence
        if i == 0:
            prob += log(start_prob[p1])
        else:
            prob += log(trans_prob[path[i-1]][p1])
        prob += log(emit_prob[p1][s1])

    # Transition to End (only possible from 'I' as in Nature Primer)
    last_state = path[-1]
    if last_state == 'I':
        prob += log(trans_prob['I']['end'])  # Transition from 'I' to End

    return prob

# Define all the Parameters as in Nature Primer
states = ['E', '5', 'I']
start_prob = {'E': 1.0, '5': 0.0, 'I': 0.0}

trans_prob = {
    'E': {'E': 0.9, '5': 0.1},  # Transition probabilities from 'E'
    '5': {'I': 1.0},             # Transition probability from '5' to 'I'
    'I': {'I': 0.9, 'end': 0.1},  # Transition probabilities from 'I' to 'I' and 'I' to 'End'
}

emit_prob = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},  # Emission probabilities for 'E'
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},    # Emission probabilities for '5'
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}        # Emission probabilities for 'I'
}

# Viterbi Algorithm implementation to find the most likely path
def viterbi(obs_seq):
    # Initialize Viterbi matrix and path matrix
    V = [{}]
    path = {}

    # Initialization step for the first observation
    for state in states:
        V[0][state] = log(start_prob[state]) + log(emit_prob[state][obs_seq[0]])
        path[state] = [state]

    # Recursion step for all subsequent observations
    for t in range(1, len(obs_seq)):
        V.append({})
        new_path = {}

        for curr_state in states:
            max_prob, prev_state_best = max(
                (V[t-1][prev_state] + log(trans_prob[prev_state].get(curr_state, 0)) + log(emit_prob[curr_state].get(obs_seq[t], 0)), prev_state)
                for prev_state in states
            )
            V[t][curr_state] = max_prob
            new_path[curr_state] = path[prev_state_best] + [curr_state]

        path = new_path

    # Termination step: Find the most probable final state
    max_prob, best_final_state = max(
        (V[len(obs_seq)-1][state], state) for state in states
    )

    # Return the best path and its probability
    return path[best_final_state], max_prob

# Test with a given sequence
observed_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

best_path, best_prob = viterbi(observed_sequence)

# Convert the path to a string for readability
best_path_str = ''.join(best_path)

# Print results
print(f"Most likely path: {best_path_str}")
print(f"Log probability of this path: {round(best_prob, 2)}")

Most likely path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability of this path: -38.68
